# 🚕 NYC Taxi Trip Duration Prediction
## End-to-End ML Pipeline with Google BigQuery

**Dataset:** NYC TLC Yellow Taxi Trips 2022  
**Goal:** Predict trip duration in minutes  
**Models:** Linear Regression, Random Forest, XGBoost

In [ ]:
import os, sys
sys.path.append("..")  # so it can find project_config.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from google.cloud import bigquery
from google.oauth2 import service_account
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import joblib

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"path\to\your\key.json"
os.environ["GCP_PROJECT_ID"] = "your-project-id"

from project_config import DATA_DIR, MODEL_DIR, TARGET_COLUMN, FEATURE_COLUMNS

ModuleNotFoundError: No module named 'project_config'

## Step 1 — Connect to BigQuery & Load Data

In [ ]:
# Load already-downloaded data (run 01_data_connection.py first)
df_raw = pd.read_parquet(os.path.join(DATA_DIR, "raw_taxi_data.parquet"))
print(f"Shape: {df_raw.shape}")
df_raw.head()

## Step 2 — Data Cleaning

In [ ]:
df = pd.read_parquet(os.path.join(DATA_DIR, "clean_taxi_data.parquet"))
print(f"Clean shape: {df.shape}")
df.describe()

## Step 3 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[TARGET_COLUMN], bins=60, kde=True, ax=axes[0])
axes[0].set_title("Trip Duration Distribution")
sns.histplot(np.log1p(df[TARGET_COLUMN]), bins=60, kde=True, ax=axes[1], color="coral")
axes[1].set_title("Log-Transformed Duration")
plt.tight_layout()
plt.show()

In [ ]:
num_cols = [TARGET_COLUMN, "trip_distance", "passenger_count", "fare_amount", "pickup_hour"]
corr = df[num_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.show()

## Step 4 — Feature Engineering & Train/Test Split

In [ ]:
X_train = pd.read_parquet(os.path.join(DATA_DIR, "X_train.parquet"))
X_val   = pd.read_parquet(os.path.join(DATA_DIR, "X_val.parquet"))
X_test  = pd.read_parquet(os.path.join(DATA_DIR, "X_test.parquet"))
y_train = pd.read_parquet(os.path.join(DATA_DIR, "y_train.parquet")).squeeze()
y_val   = pd.read_parquet(os.path.join(DATA_DIR, "y_val.parquet")).squeeze()
y_test  = pd.read_parquet(os.path.join(DATA_DIR, "y_test.parquet")).squeeze()

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
X_train.head()

## Step 5 — Model Comparison

In [ ]:
results = pd.read_csv(os.path.join(DATA_DIR, "model_comparison.csv"), index_col="model")
results.sort_values("RMSE")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, metric in zip(axes, ["RMSE", "MAE", "R2"]):
    results[metric].sort_values().plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(metric)
    ax.set_xlabel(metric)
plt.suptitle("Model Comparison", fontweight="bold")
plt.tight_layout()
plt.show()

## Step 6 — Final Evaluation (Test Set)

In [ ]:
model  = joblib.load(os.path.join(MODEL_DIR, "xgboost.joblib"))
y_pred = model.predict(X_test)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"{'='*40}")
print(f"  FINAL TEST SET RESULTS (XGBoost)")
print(f"{'='*40}")
print(f"  MSE  : {mse:.4f}")
print(f"  RMSE : {rmse:.4f} minutes")
print(f"  MAE  : {mae:.4f} minutes")
print(f"  R²   : {r2:.4f}")
print(f"{'='*40}")

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.2, s=5, color="steelblue")
lim = [0, max(y_test.max(), y_pred.max())]
plt.plot(lim, lim, "r--", lw=1.5, label="Perfect prediction")
plt.xlabel("Actual Duration (min)")
plt.ylabel("Predicted Duration (min)")
plt.title("Predicted vs Actual — XGBoost")
plt.legend()
plt.show()

In [ ]:
importance = model.get_booster().get_score(importance_type="gain")
imp_df = pd.Series(importance).sort_values()
plt.figure(figsize=(8, 6))
imp_df.plot(kind="barh", color="steelblue")
plt.title("XGBoost Feature Importance (Gain)")
plt.tight_layout()
plt.show()